# LeetCode 1573 — Number of Ways to Split a String bằng Python

Link đề: https://leetcode.com/problems/number-of-ways-to-split-a-string/ · Độ khó: **Medium** · Chủ đề: Math, String

Skeleton LeetCode:

```python
class Solution:
    def numWays(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hoá "cắt chuỗi thành 3 phần không rỗng" bằng **hai vị trí cắt** `1 ≤ i < j ≤ n-1`, và viết baseline duyệt mọi cặp `(i, j)` với **mảng tiền tố** đếm số `'1'`.
2. Nhận ra điều kiện cần: tổng số `'1'` phải **chia hết cho 3**; mỗi phần có đúng `k = tổng/3` số `'1'`.
3. Đếm số cách bằng **nguyên lý nhân**: số chỗ đặt nhát cắt thứ nhất × số chỗ đặt nhát cắt thứ hai, mỗi số bằng "độ rộng khoảng trống" giữa hai số `'1'` ở ranh giới.
4. Xử lý riêng trường hợp **toàn số 0**: đáp án là tổ hợp `C(n-1, 2)`, và lấy **modulo `10⁹ + 7`** đúng chỗ.
5. Mở rộng sang follow-up: mảng số nguyên bất kỳ, đếm cách chia thành 3 đoạn có **tổng bằng nhau** bằng đếm tiền tố.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` (chỉ gồm `'0'` và `'1'`). Ta cắt `s` thành **3 chuỗi con không rỗng** `s1`, `s2`, `s3` sao cho `s1 + s2 + s3 = s`.

Trả về **số cách cắt** sao cho **số lượng ký tự `'1'`** trong `s1`, `s2`, `s3` **bằng nhau**. Vì đáp án có thể rất lớn, trả về theo **modulo `10⁹ + 7`**.

**Ví dụ 1:** `s = "10101"` → `4`: `"1|010|1"`, `"1|01|01"`, `"10|10|1"`, `"10|1|01"`.

**Ví dụ 2:** `s = "1001"` → `0` (có 2 số `'1'`, không chia đều cho 3 phần).

**Ví dụ 3:** `s = "0000"` → `3`: `"0|0|00"`, `"0|00|0"`, `"00|0|0"`.

**Ràng buộc:**
- `3 ≤ s.length ≤ 10⁵`
- `s[i]` là `'0'` hoặc `'1'`.

In [1]:
import random
import time
from bisect import bisect_left, bisect_right
from itertools import accumulate
from math import comb

MOD = 10**9 + 7
examples = [("10101", 4), ("1001", 0), ("0000", 3)]
for s, want in examples:
    print(f"s = {s!r} -> mong đợi {want}")

s = '10101' -> mong đợi 4
s = '1001' -> mong đợi 0
s = '0000' -> mong đợi 3


## 3. Kiến thức nền

### 3.1 Cắt chuỗi bằng slicing và hai vị trí cắt

Một cách chia `s` thành `s[:i] | s[i:j] | s[j:]` được xác định bởi hai **vị trí cắt** `i < j`. Ba phần không rỗng ⇔ `1 ≤ i < j ≤ n-1`.

In [2]:
s = "10101"
n = len(s)
for i in range(1, n - 1):
    for j in range(i + 1, n):
        print(f"i={i}, j={j}: {s[:i]!r:>6} | {s[i:j]!r:<6} | {s[j:]!r}")

i=1, j=2:    '1' | '0'    | '101'
i=1, j=3:    '1' | '01'   | '01'
i=1, j=4:    '1' | '010'  | '1'
i=2, j=3:   '10' | '1'    | '01'
i=2, j=4:   '10' | '10'   | '1'
i=3, j=4:  '101' | '0'    | '1'


### 3.2 Mảng tiền tố (prefix sum) để đếm số `'1'` trong đoạn

`pre[i]` = số `'1'` trong `s[:i]`. Khi đó số `'1'` trong `s[a:b]` là `pre[b] - pre[a]`, tính trong `O(1)`.

`itertools.accumulate` giúp tạo mảng tiền tố gọn.

In [3]:
s = "10101"
pre = [0] + list(accumulate(int(c) for c in s))
print("pre =", pre)
print("số '1' trong s[1:4] =", pre[4] - pre[1], "| kiểm tra:", s[1:4].count("1"))

pre = [0, 1, 1, 2, 2, 3]
số '1' trong s[1:4] = 1 | kiểm tra: 1


### 3.3 Nguyên lý nhân và tổ hợp `C(n, 2)`

- **Nguyên lý nhân:** nếu lựa chọn thứ nhất có `a` cách, lựa chọn thứ hai (độc lập) có `b` cách, thì có `a · b` cách tổng cộng.
- **Tổ hợp:** chọn 2 phần tử phân biệt từ `m` phần tử có `C(m, 2) = m(m-1)/2` cách. Python có `math.comb`.

In [4]:
print("C(3, 2) =", comb(3, 2), "| C(4, 2) =", comb(4, 2), "| công thức m(m-1)//2 với m=4:", 4 * 3 // 2)
# Chuỗi "0000": có n-1 = 3 vị trí cắt (giữa các ký tự), chọn 2 trong 3
print("số cách cho '0000' =", comb(len("0000") - 1, 2))

C(3, 2) = 3 | C(4, 2) = 6 | công thức m(m-1)//2 với m=4: 6
số cách cho '0000' = 3


### 3.4 Modulo `10⁹ + 7`

Python không tràn số, nhưng đề yêu cầu trả về `ans % MOD`. Với tích hai số, có thể lấy modulo một lần ở cuối (Python) hoặc `(a % MOD) * (b % MOD) % MOD` (ngôn ngữ có tràn số).

In [5]:
n = 10**5
raw = (n - 1) * (n - 2) // 2
print("C(n-1, 2) thô =", raw, "| sau modulo =", raw % MOD)

C(n-1, 2) thô = 4999850001 | sau modulo = 999849973


### 3.5 Lấy danh sách vị trí của các ký tự `'1'`

`[i for i, c in enumerate(s) if c == "1"]` cho danh sách chỉ số (đã tăng dần) của các số `'1'`.

In [6]:
s = "100100010100110"
ones = [i for i, c in enumerate(s) if c == "1"]
print("vị trí các '1':", ones, "| tổng số '1':", len(ones))

vị trí các '1': [0, 3, 7, 9, 12, 13] | tổng số '1': 6


## 4. Ý tưởng nền tảng / trực giác

Lấy `s = "10101"` (`n = 5`, có 3 số `'1'` ở vị trí `0, 2, 4`). Mỗi phần phải có đúng `k = 1` số `'1'`.

- **Nhát cắt 1** (vị trí `i`): `s1 = s[:i]` phải chứa đúng 1 số `'1'` ⇒ `s1` chứa số `'1'` ở vị trí 0 nhưng **không** chứa số `'1'` ở vị trí 2 ⇒ `i ∈ {1, 2}` (cắt ngay sau vị trí 0 hoặc sau vị trí 1). Có `2 - 0 = 2` lựa chọn.
- **Nhát cắt 2** (vị trí `j`): `s1 + s2 = s[:j]` phải chứa đúng 2 số `'1'` ⇒ chứa vị trí 2 nhưng không chứa vị trí 4 ⇒ `j ∈ {3, 4}`. Có `4 - 2 = 2` lựa chọn.
- Hai lựa chọn **độc lập** (mọi `i ∈ {1,2}` đều nhỏ hơn mọi `j ∈ {3,4}`) ⇒ `2 × 2 = 4` cách. ✔

Tổng quát: nhát cắt 1 nằm trong "khoảng trống" giữa số `'1'` thứ `k` và thứ `k+1`; nhát cắt 2 nằm giữa số `'1'` thứ `2k` và thứ `2k+1`. Số chỗ = hiệu hai chỉ số.

Trường hợp `s = "0000"`: không có số `'1'`, **mọi** cặp `i < j` đều hợp lệ ⇒ chọn 2 trong `n - 1 = 3` vị trí cắt ⇒ `3`.

In [7]:
s = "10101"
ones = [i for i, c in enumerate(s) if c == "1"]
k = len(ones) // 3
gap1 = ones[k] - ones[k - 1]          # giữa '1' thứ k và thứ k+1 (đánh số từ 1)
gap2 = ones[2 * k] - ones[2 * k - 1]  # giữa '1' thứ 2k và thứ 2k+1
print("ones =", ones, "| k =", k, "| gap1 =", gap1, "| gap2 =", gap2, "| đáp án =", gap1 * gap2)

ones = [0, 2, 4] | k = 1 | gap1 = 2 | gap2 = 2 | đáp án = 4


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** duyệt mọi cặp vị trí cắt `1 ≤ i < j ≤ n-1`, đếm số `'1'` của 3 phần và kiểm tra bằng nhau.

- Bản ngây thơ đếm bằng `count` mỗi lần: `O(n³)`.
- Bản dùng **mảng tiền tố**: mỗi cặp kiểm tra `O(1)` ⇒ `O(n²)`.

In [8]:
def brute_force_naive(s: str) -> int:
    n, cnt = len(s), 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            a, b, c = s[:i].count("1"), s[i:j].count("1"), s[j:].count("1")
            if a == b == c:
                cnt += 1
    return cnt % MOD

def brute_force(s: str) -> int:
    n = len(s)
    pre = [0] * (n + 1)
    for i, ch in enumerate(s):
        pre[i + 1] = pre[i] + (ch == "1")
    cnt = 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            a, b, c = pre[i], pre[j] - pre[i], pre[n] - pre[j]
            if a == b == c:
                cnt += 1
    return cnt % MOD

for s, want in examples:
    print(s, "->", brute_force_naive(s), brute_force(s), "| mong đợi", want)

10101 -> 4 4 | mong đợi 4
1001 -> 0 0 | mong đợi 0
0000 -> 3 3 | mong đợi 3


**Độ phức tạp:** `O(n²)` thời gian (bản tiền tố), `O(n)` bộ nhớ.

**Nút thắt:** `n = 10⁵` ⇒ khoảng `5·10⁹` cặp — quá chậm. Ta cần đếm số cặp hợp lệ **mà không liệt kê** từng cặp.

In [9]:
rng = random.Random(1)
s_big = "".join(rng.choice("01") for _ in range(600))
t0 = time.perf_counter()
r = brute_force(s_big)
print(f"n=600: kết quả {r}, mất {time.perf_counter() - t0:.3f}s -> n=10^5 sẽ chậm gấp ~27000 lần")

n=600: kết quả 0, mất 0.012s -> n=10^5 sẽ chậm gấp ~27000 lần


## 6. Ý tưởng thuật toán / bước đột phá

Gọi `t` = tổng số `'1'` trong `s`.

**Quy tắc 1:** Nếu `t % 3 != 0` thì **không có cách nào** ⇒ trả `0`.

**Quy tắc 2:** Nếu `t == 0` thì mọi cặp `(i, j)` với `1 ≤ i < j ≤ n-1` đều hợp lệ ⇒ đáp án `C(n-1, 2) = (n-1)(n-2)/2`.

**Quy tắc 3:** Nếu `t = 3k` với `k ≥ 1`. Gọi `p₁ < p₂ < … < p_t` là vị trí các số `'1'` (đánh số từ 1).
- `s[:i]` chứa đúng `k` số `'1'` ⇔ `p_k < i ≤ p_{k+1}` ⇔ có `p_{k+1} - p_k` giá trị `i`.
- `s[:j]` chứa đúng `2k` số `'1'` ⇔ `p_{2k} < j ≤ p_{2k+1}` ⇔ có `p_{2k+1} - p_{2k}` giá trị `j`.
- Mọi `i` hợp lệ `≤ p_{k+1} ≤ p_{2k} < j` ⇒ luôn `i < j`, phần giữa và phần cuối tự động không rỗng và đều có `k` số `'1'`.
- ⇒ **đáp án = (p_{k+1} - p_k) × (p_{2k+1} - p_{2k})**.

Dễ nhớ: "số chỗ đặt dao ở mỗi ranh giới = số số 0 giữa hai số 1 kề ranh giới **cộng 1**".

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Danh sách vị trí số `'1'` + nguyên lý nhân

**Đoạn 1:** thu thập `ones` = danh sách chỉ số các `'1'`; `t = len(ones)`.

**Đoạn 2:** xử lý `t % 3 != 0` → `0`; `t == 0` → `C(n-1, 2) % MOD`.

**Đoạn 3:** `k = t // 3`. Với chỉ số **từ 0** trong Python: số `'1'` thứ `k` là `ones[k-1]`, thứ `k+1` là `ones[k]`. Vậy `gap1 = ones[k] - ones[k-1]`, `gap2 = ones[2k] - ones[2k-1]`.

In [10]:
def num_ways_positions(s: str) -> int:
    ones = [i for i, c in enumerate(s) if c == "1"]
    t, n = len(ones), len(s)
    if t % 3:
        return 0
    if t == 0:
        return (n - 1) * (n - 2) // 2 % MOD
    k = t // 3
    gap1 = ones[k] - ones[k - 1]
    gap2 = ones[2 * k] - ones[2 * k - 1]
    return gap1 * gap2 % MOD

for s, want in examples:
    print(s, "->", num_ways_positions(s), "| mong đợi", want)

10101 -> 4 | mong đợi 4
1001 -> 0 | mong đợi 0
0000 -> 3 | mong đợi 3


Code nộp LeetCode cho cách 2:

```python
class Solution:
    def numWays(self, s: str) -> int:
        MOD = 10**9 + 7
        ones = [i for i, c in enumerate(s) if c == "1"]
        t, n = len(ones), len(s)
        if t % 3:
            return 0
        if t == 0:
            return (n - 1) * (n - 2) // 2 % MOD
        k = t // 3
        return (ones[k] - ones[k - 1]) * (ones[2 * k] - ones[2 * k - 1]) % MOD
```

**Độ phức tạp:** `O(n)` thời gian, `O(t)` bộ nhớ cho danh sách vị trí (tệ nhất `O(n)`).

### 7.2 Cách 3 — Đếm tiền tố, bộ nhớ `O(1)` phụ

Không cần lưu vị trí. Duyệt các vị trí cắt `c = 1 … n-1` và giữ `cur = số '1' trong s[:c]`:
- `cnt1` = số vị trí cắt có `cur == k` (ứng viên cho nhát cắt 1);
- `cnt2` = số vị trí cắt có `cur == 2k` (ứng viên cho nhát cắt 2).

Khi `k ≥ 1`, mọi vị trí có `cur == k` đều đứng **trước** mọi vị trí có `cur == 2k` (vì `cur` không giảm và `k < 2k`), nên đáp án là `cnt1 × cnt2`. Đây chính là cách 2 nhìn theo góc khác: `cnt1 = gap1`, `cnt2 = gap2`.

Khi `k = 0` thì `cnt1 = cnt2 = n-1` nhưng cặp phải có `i < j` ⇒ không dùng được tích, quay về `C(n-1, 2)`.

In [11]:
def num_ways_counting(s: str) -> int:
    n = len(s)
    t = s.count("1")
    if t % 3:
        return 0
    if t == 0:
        return (n - 1) * (n - 2) // 2 % MOD
    k = t // 3
    cur = cnt1 = cnt2 = 0
    for c in range(1, n):          # vị trí cắt c: tách s[:c] | s[c:]
        cur += s[c - 1] == "1"
        if cur == k:
            cnt1 += 1
        elif cur == 2 * k:
            cnt2 += 1
    return cnt1 * cnt2 % MOD

for s, want in examples:
    print(s, "->", num_ways_counting(s), "| mong đợi", want)

10101 -> 4 | mong đợi 4
1001 -> 0 | mong đợi 0
0000 -> 3 | mong đợi 3


Code nộp LeetCode cho cách 3:

```python
class Solution:
    def numWays(self, s: str) -> int:
        MOD = 10**9 + 7
        n, t = len(s), s.count("1")
        if t % 3:
            return 0
        if t == 0:
            return (n - 1) * (n - 2) // 2 % MOD
        k = t // 3
        cur = cnt1 = cnt2 = 0
        for c in range(1, n):
            cur += s[c - 1] == "1"
            if cur == k:
                cnt1 += 1
            elif cur == 2 * k:
                cnt2 += 1
        return cnt1 * cnt2 % MOD
```

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

### 7.3 Cách 4 — Tổng quát hoá (follow-up của hint): mảng số nguyên bất kỳ

Hint 3 của đề: *"Tổng quát với các số trong `[-10⁹, 10⁹]` sao cho tổng ba đoạn bằng nhau?"* Lúc này **không** còn tính đơn điệu (số âm làm tiền tố giảm), nên không dùng "khoảng trống" được.

**Ý tưởng đếm tiền tố tổng quát:** với tổng `S`, nếu `S % 3 != 0` → `0`. Ngược lại `T = S/3`. Một cặp cắt `(i, j)` hợp lệ ⇔ `pre[i] = T` và `pre[j] = 2T` với `1 ≤ i < j ≤ n-1`.

Duyệt `j` từ `2` đến `n-1`; trước khi xét `j`, cập nhật `seen` = số `i < j` (với `i ≥ 1`) có `pre[i] = T`. Nếu `pre[j] = 2T` thì cộng `seen`. Cách này đúng cả khi `T = 0` (tự động thành tổ hợp) nên **không cần trường hợp đặc biệt**.

In [12]:
def count_equal_sum_splits(nums) -> int:
    n = len(nums)
    pre = [0] + list(accumulate(nums))
    S = pre[n]
    if S % 3:
        return 0
    T = S // 3
    seen = ans = 0
    for j in range(2, n):
        if pre[j - 1] == T:        # i = j-1 trở thành ứng viên cho nhát cắt 1
            seen += 1
        if pre[j] == 2 * T:
            ans += seen
    return ans

def num_ways_general(s: str) -> int:
    return count_equal_sum_splits([int(c) for c in s]) % MOD

for s, want in examples:
    print(s, "->", num_ways_general(s), "| mong đợi", want)
print("mảng có số âm [1,-1,0,2,-2,0] ->", count_equal_sum_splits([1, -1, 0, 2, -2, 0]))

10101 -> 4 | mong đợi 4
1001 -> 0 | mong đợi 0
0000 -> 3 | mong đợi 3
mảng có số âm [1,-1,0,2,-2,0] -> 3


Code nộp LeetCode dùng cách 4 (đúng chữ ký, không cần nhánh `t == 0`):

```python
class Solution:
    def numWays(self, s: str) -> int:
        MOD = 10**9 + 7
        n = len(s)
        pre = [0] * (n + 1)
        for i, c in enumerate(s):
            pre[i + 1] = pre[i] + (c == "1")
        if pre[n] % 3:
            return 0
        T = pre[n] // 3
        seen = ans = 0
        for j in range(2, n):
            if pre[j - 1] == T:
                seen += 1
            if pre[j] == 2 * T:
                ans += seen
        return ans % MOD
```

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (mảng tiền tố; có thể bỏ mảng bằng cách giữ hai biến tiền tố).

## 8. TRACE từng bước

Ví dụ `s = "1010010001"`: vị trí `'1'` là `0, 2, 5, 9` ⇒ `t = 4` ⇒ không chia hết cho 3 ⇒ `0`. Lấy ví dụ khác có `t = 6`: `s = "100100010100110"`.

`ones = [0, 3, 7, 9, 12, 13]`, `k = 2`.
- `gap1 = ones[2] - ones[1] = 7 - 3 = 4` (nhát cắt 1 đặt ở vị trí `4, 5, 6, 7`).
- `gap2 = ones[4] - ones[3] = 12 - 9 = 3` (nhát cắt 2 đặt ở `10, 11, 12`).
- Đáp án `4 × 3 = 12`.

Bảng cách 3 (chỉ các vị trí đáng chú ý):

| c | s[c-1] | cur | cur == k? | cur == 2k? | cnt1 | cnt2 |
|---|---|---|---|---|---|---|
| 1..3 | | 1 | | | 0 | 0 |
| 4 | 1 | 2 | ✔ | | 1 | 0 |
| 5,6,7 | 0 | 2 | ✔ | | 4 | 0 |
| 8,9 | 1,0 | 3 | | | 4 | 0 |
| 10 | 1 | 4 | | ✔ | 4 | 1 |
| 11,12 | 0 | 4 | | ✔ | 4 | 3 |
| 13,14 | 1,1 | 5,6 | | | 4 | 3 |

In [13]:
def trace_counting(s: str):
    n, t = len(s), s.count("1")
    print(f"s={s!r}, n={n}, tổng số '1' t={t}")
    if t % 3:
        print("t không chia hết cho 3 -> 0")
        return 0
    if t == 0:
        print("toàn 0 -> C(n-1, 2)")
        return (n - 1) * (n - 2) // 2 % MOD
    k = t // 3
    print("k =", k)
    cur = cnt1 = cnt2 = 0
    for c in range(1, n):
        cur += s[c - 1] == "1"
        mark = ""
        if cur == k:
            cnt1 += 1
            mark = "<- ứng viên cắt 1"
        elif cur == 2 * k:
            cnt2 += 1
            mark = "<- ứng viên cắt 2"
        print(f"  c={c:>2}: {s[:c]:<15}| cur={cur} cnt1={cnt1} cnt2={cnt2} {mark}")
    print("đáp án =", cnt1, "x", cnt2, "=", cnt1 * cnt2)
    return cnt1 * cnt2 % MOD

trace_counting("100100010100110")

s='100100010100110', n=15, tổng số '1' t=6
k = 2
  c= 1: 1              | cur=1 cnt1=0 cnt2=0 
  c= 2: 10             | cur=1 cnt1=0 cnt2=0 
  c= 3: 100            | cur=1 cnt1=0 cnt2=0 
  c= 4: 1001           | cur=2 cnt1=1 cnt2=0 <- ứng viên cắt 1
  c= 5: 10010          | cur=2 cnt1=2 cnt2=0 <- ứng viên cắt 1
  c= 6: 100100         | cur=2 cnt1=3 cnt2=0 <- ứng viên cắt 1
  c= 7: 1001000        | cur=2 cnt1=4 cnt2=0 <- ứng viên cắt 1
  c= 8: 10010001       | cur=3 cnt1=4 cnt2=0 
  c= 9: 100100010      | cur=3 cnt1=4 cnt2=0 
  c=10: 1001000101     | cur=4 cnt1=4 cnt2=1 <- ứng viên cắt 2
  c=11: 10010001010    | cur=4 cnt1=4 cnt2=2 <- ứng viên cắt 2
  c=12: 100100010100   | cur=4 cnt1=4 cnt2=3 <- ứng viên cắt 2
  c=13: 1001000101001  | cur=5 cnt1=4 cnt2=3 
  c=14: 10010001010011 | cur=6 cnt1=4 cnt2=3 
đáp án = 4 x 3 = 12


12

In [14]:
trace_counting("10101")
print()
trace_counting("1010010001")
print()
trace_counting("00000")

s='10101', n=5, tổng số '1' t=3
k = 1
  c= 1: 1              | cur=1 cnt1=1 cnt2=0 <- ứng viên cắt 1
  c= 2: 10             | cur=1 cnt1=2 cnt2=0 <- ứng viên cắt 1
  c= 3: 101            | cur=2 cnt1=2 cnt2=1 <- ứng viên cắt 2
  c= 4: 1010           | cur=2 cnt1=2 cnt2=2 <- ứng viên cắt 2
đáp án = 2 x 2 = 4

s='1010010001', n=10, tổng số '1' t=4
t không chia hết cho 3 -> 0

s='00000', n=5, tổng số '1' t=0
toàn 0 -> C(n-1, 2)


6

## 9. Code nộp LeetCode

In [15]:
class Solution:
    def numWays(self, s: str) -> int:
        MOD = 10**9 + 7
        ones = [i for i, c in enumerate(s) if c == "1"]
        t, n = len(ones), len(s)
        if t % 3:
            return 0
        if t == 0:
            return (n - 1) * (n - 2) // 2 % MOD
        k = t // 3
        return (ones[k] - ones[k - 1]) * (ones[2 * k] - ones[2 * k - 1]) % MOD

sol = Solution()
for s, want in examples:
    print(s, "->", sol.numWays(s), "| mong đợi", want)

10101 -> 4 | mong đợi 4
1001 -> 0 | mong đợi 0
0000 -> 3 | mong đợi 3


## 10. Vì sao thuật toán đúng

Gọi `pre(c)` = số `'1'` trong `s[:c]`, là hàm **không giảm** theo `c`, tăng đúng 1 tại mỗi vị trí ngay sau một số `'1'`.

1. Nếu ba phần có cùng số `'1'` = `x` thì `3x = t` ⇒ `t` chia hết cho 3 (quy tắc 1).
2. Với `t = 3k, k ≥ 1`: cặp `(i, j)` hợp lệ ⇔ `pre(i) = k` và `pre(j) = 2k` (vì khi đó phần giữa có `2k - k = k`, phần cuối `3k - 2k = k`). Tập `{c : pre(c) = k}` là một **đoạn số nguyên liên tiếp** `(p_k, p_{k+1}]` có độ dài `p_{k+1} - p_k ≥ 1`; tương tự `{c : pre(c) = 2k} = (p_{2k}, p_{2k+1}]`. Hai đoạn rời nhau và đoạn đầu đứng trước (vì `k < 2k`), nên mọi tổ hợp đều có `i < j`, và `1 ≤ i`, `j ≤ n-1` tự thoả (`i > p_k ≥ 0`, `j ≤ p_{2k+1} ≤ n-1`). Số cặp = tích độ dài.
3. Với `t = 0`: `pre ≡ 0`, mọi cặp `1 ≤ i < j ≤ n-1` hợp lệ ⇒ `C(n-1, 2)`.

Cách 4 đếm trực tiếp số cặp `(i<j)` với `pre(i) = T`, `pre(j) = 2T` theo định nghĩa nên đúng cho mọi mảng số nguyên.

## 11. Phân tích Time / Space

Gọi `n = len(s)`, `t` = số `'1'`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force (count) | `O(n³)` | `O(n)` |
| 1'. Brute force + tiền tố | `O(n²)` | `O(n)` |
| 2. Vị trí `'1'` + nhân | `O(n)` | `O(t)` |
| 3. Đếm tiền tố | `O(n)` | `O(1)` |
| 4. Tổng quát (tổng bằng nhau) | `O(n)` | `O(n)` (hoặc `O(1)`) |

Mọi cách tuyến tính đều phải đọc hết chuỗi ít nhất một lần, nên `O(n)` là tối ưu. Không có khác biệt giữa average và worst.

## 12. Bộ test

In [16]:
ALL = [brute_force, num_ways_positions, num_ways_counting, num_ways_general, Solution().numWays]

# 12.1 Ví dụ đề
for f in ALL:
    for s, want in examples:
        assert f(s) == want, (f.__name__, s)
print("Ví dụ đề: OK")

# 12.2 Edge case
edge = [
    ("000", 1),          # n = 3 nhỏ nhất, toàn 0: chỉ "0|0|0"
    ("111", 1),          # mỗi phần đúng một '1'
    ("100", 0),          # t = 1
    ("111111", 1),       # k = 2, không có khoảng trống thừa
    ("110110110", 4),    # k = 2, mỗi ranh giới có 2 chỗ cắt
    ("10010010", 9),     # gap1 = 3, gap2 = 3
    ("0101010", 4),      # số 0 ở hai đầu không ảnh hưởng
]
for f in ALL:
    for s, want in edge:
        assert f(s) == want, (f.__name__, s, f(s), want)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
# 12.3 Ngẫu nhiên, seed cố định: đối chiếu mọi cách với brute force naive
rng = random.Random(1573)
for _ in range(1500):
    n = rng.randint(3, 18)
    p1 = rng.choice([0.0, 0.2, 0.5, 0.8, 1.0])
    s = "".join("1" if rng.random() < p1 else "0" for _ in range(n))
    want = brute_force_naive(s)
    for f in ALL:
        assert f(s) == want, (f.__name__, s)
print("1500 chuỗi ngẫu nhiên: OK")

1500 chuỗi ngẫu nhiên: OK


In [18]:
# 12.4 Test lớn đúng ràng buộc + kiểm tra modulo
big_zero = "0" * 10**5
want = comb(10**5 - 1, 2) % MOD
for f in [num_ways_positions, num_ways_counting, num_ways_general, Solution().numWays]:
    assert f(big_zero) == want
print("toàn 0, n=10^5 ->", want, "(C(n-1,2) =", comb(10**5 - 1, 2), "> MOD nên phải modulo)")

big = "1" + "0" * 49998 + "1" + "0" * 49998 + "1"
print("ba số 1 cách xa ->", Solution().numWays(big), "| kỳ vọng", 49999 * 49999 % MOD)
assert Solution().numWays(big) == 49999 * 49999 % MOD

rng = random.Random(9)
s = "".join(rng.choice("01") for _ in range(10**5))
t0 = time.perf_counter()
res = [f(s) for f in [num_ways_positions, num_ways_counting, num_ways_general]]
assert len(set(res)) == 1
print(f"chuỗi ngẫu nhiên n=10^5: {res[0]}, 3 cách mất {time.perf_counter() - t0:.3f}s")

toàn 0, n=10^5 -> 999849973 (C(n-1,2) = 4999850001 > MOD nên phải modulo)
ba số 1 cách xa -> 499899987 | kỳ vọng 499899987


chuỗi ngẫu nhiên n=10^5: 3, 3 cách mất 0.030s


In [19]:
# 12.5 Cách 4 trên mảng số nguyên có số âm, đối chiếu brute force
def brute_equal_sum(nums):
    n, cnt = len(nums), 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            if sum(nums[:i]) == sum(nums[i:j]) == sum(nums[j:]):
                cnt += 1
    return cnt

rng = random.Random(15)
for _ in range(800):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(3, 12))]
    assert count_equal_sum_splits(nums) == brute_equal_sum(nums), nums
print("Tổng quát hoá với số âm: OK")

Tổng quát hoá với số âm: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp toàn số 0

Công thức khoảng trống truy cập `ones[k-1]` với `k = 0` ⇒ `ones[-1]` (chỉ số âm, Python không báo lỗi!) hoặc `IndexError` khi `ones` rỗng.

In [20]:
def wrong_no_zero_case(s):
    ones = [i for i, c in enumerate(s) if c == "1"]
    t = len(ones)
    if t % 3:
        return 0
    k = t // 3
    return (ones[k] - ones[k - 1]) * (ones[2 * k] - ones[2 * k - 1]) % MOD   # SAI khi t == 0

for s in ["0000", "00000"]:
    try:
        print(s, "-> sai:", wrong_no_zero_case(s))
    except IndexError as e:
        print(s, "-> IndexError:", e, "| đúng:", Solution().numWays(s))

0000 -> IndexError: list index out of range | đúng: 3
00000 -> IndexError: list index out of range | đúng: 6


### Lỗi 2 — Đếm số số 0 trong khoảng trống thay vì "số 0 + 1"

Số chỗ đặt dao giữa hai số `'1'` ở vị trí `a < b` là `b - a` (gồm cả chỗ ngay sau `a`), **không phải** `b - a - 1`.

In [21]:
def wrong_gap(s):
    ones = [i for i, c in enumerate(s) if c == "1"]
    t, n = len(ones), len(s)
    if t % 3:
        return 0
    if t == 0:
        return (n - 1) * (n - 2) // 2 % MOD
    k = t // 3
    return (ones[k] - ones[k - 1] - 1) * (ones[2 * k] - ones[2 * k - 1] - 1) % MOD   # SAI: -1

for s in ["10101", "111"]:
    print(s, "-> sai:", wrong_gap(s), "| đúng:", Solution().numWays(s))

10101 -> sai: 1 | đúng: 4
111 -> sai: 0 | đúng: 1


### Lỗi 3 — Dùng tích `cnt1 * cnt2` cả khi `k = 0`

Khi `k = 0`, mọi vị trí đều thoả `cur == 0 == 2k` nên cặp `(i, j)` bị đếm cả `i ≥ j`.

In [22]:
def wrong_product_zero(s):
    n, t = len(s), s.count("1")
    if t % 3:
        return 0
    k = t // 3
    cur = cnt1 = cnt2 = 0
    for c in range(1, n):
        cur += s[c - 1] == "1"
        if cur == k:
            cnt1 += 1
        if cur == 2 * k:          # SAI khi k = 0: cùng một vị trí được đếm ở cả hai
            cnt2 += 1
    return cnt1 * cnt2 % MOD

print("0000 -> sai:", wrong_product_zero("0000"), "| đúng:", Solution().numWays("0000"))
print("10101 -> (không sai khi k>0):", wrong_product_zero("10101"))

0000 -> sai: 9 | đúng: 3
10101 -> (không sai khi k>0): 4


### Lỗi 4 — Quên modulo (hoặc chia trước khi modulo sai cách)

Python trả về số rất lớn — LeetCode sẽ báo **Wrong Answer**. Ngoài ra, nếu lấy modulo **trước** rồi mới chia `// 2` thì sai: `((n-1)(n-2) % MOD) // 2` không bằng `C(n-1,2) % MOD` nói chung.

In [23]:
n = 10**5
no_mod = (n - 1) * (n - 2) // 2
mod_then_div = ((n - 1) * (n - 2) % MOD) // 2
right = (n - 1) * (n - 2) // 2 % MOD
print("không modulo :", no_mod)
print("mod rồi chia :", mod_then_div, "| SAI" if mod_then_div != right else "")
print("chia rồi mod :", right, "| ĐÚNG")

không modulo : 4999850001
mod rồi chia : 499849969 | SAI
chia rồi mod : 999849973 | ĐÚNG


### Lỗi 5 — Cho phép phần rỗng (duyệt vị trí cắt từ 0 đến n)

Đề yêu cầu 3 phần **không rỗng** ⇒ `1 ≤ i < j ≤ n-1`. Duyệt `range(0, n+1)` sẽ đếm thừa.

In [24]:
def wrong_allow_empty(s):
    n, cnt = len(s), 0
    for i in range(0, n + 1):            # SAI: cho phép s1 rỗng
        for j in range(i, n + 1):        # SAI: cho phép s2/s3 rỗng
            if s[:i].count("1") == s[i:j].count("1") == s[j:].count("1"):
                cnt += 1
    return cnt

print("0000 -> sai:", wrong_allow_empty("0000"), "| đúng:", Solution().numWays("0000"))

0000 -> sai: 15 | đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chia thành m phần có tổng/số lượng bằng nhau"** ⇒ kiểm tra chia hết trước, rồi nghĩ đến **tiền tố**: ranh giới thứ `r` phải nằm ở chỗ tiền tố bằng `r·S/m`.
- Nếu dữ liệu **không âm** (như số `'1'`) ⇒ tiền tố không giảm ⇒ tập vị trí hợp lệ của mỗi ranh giới là **một đoạn liên tiếp** ⇒ nhân độ dài các đoạn.
- Nếu có **số âm** ⇒ dùng đếm "đã thấy bao nhiêu `i` có tiền tố `T`" khi duyệt `j` (cách 4).
- **Toàn 0** là trường hợp suy biến: tổ hợp `C(n-1, m-1)`.
- Kiểm tra nhanh bằng tay với `"111"` (đáp án 1), `"000"` (1), `"0000"` (3).

## 15. Biến thể của bài

1. **Chia thành `m` phần** có cùng số `'1'`: đáp án `Π_{r=1}^{m-1} (p_{rk+1} - p_{rk})`, toàn 0 thì `C(n-1, m-1)` (bài tập 4).
2. **LC 1013 — Partition Array Into Three Parts With Equal Sum:** chỉ hỏi có/không, mảng có số âm.
3. **LC 1712 — Ways to Split Array Into Three Subarrays:** điều kiện `tổng trái ≤ tổng giữa ≤ tổng phải` với số không âm ⇒ tìm nhị phân / hai con trỏ (bài tập 5).
4. **LC 2147 — Number of Ways to Divide a Long Corridor:** mỗi đoạn có đúng 2 ghế ⇒ nhân các khoảng trống giữa các cặp ghế.
5. **LC 1422 — Maximum Score After Splitting a String:** cắt 1 lần, tối ưu số 0 bên trái + số 1 bên phải.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | duyệt mọi cặp cắt, đếm bằng tiền tố | `O(n²)` | `O(n)` | kiểm thử, `n` nhỏ |
| 2. Vị trí `'1'` | nhân hai khoảng trống ở ranh giới | `O(n)` | `O(t)` | lời giải ngắn gọn, dễ giải thích |
| 3. Đếm tiền tố | đếm vị trí có tiền tố `k` và `2k` | `O(n)` | `O(1)` | khi cần tiết kiệm bộ nhớ |
| 4. Tổng quát | đếm cặp `(i<j)` có tiền tố `T`, `2T` | `O(n)` | `O(n)` | follow-up: số âm, không cần nhánh đặc biệt |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().numWays("1100110011")` trả về bao nhiêu?

**Bài 2 (tính tay):** Với `s = "0000000"` (7 số 0), đáp án là bao nhiêu? Viết công thức.

**Bài 3 (điền TODO):** Viết `ex3_cut_positions(s)` trả về **tuple hai list**: các vị trí hợp lệ cho nhát cắt 1 và cho nhát cắt 2 (giả sử `t > 0` và `t % 3 == 0`).

**Bài 4 (tự viết lại / tổng quát):** Viết `ex4_num_ways_m(s, m)` đếm số cách cắt `s` thành `m ≥ 2` phần không rỗng có cùng số `'1'` (modulo `10⁹+7`).

**Bài 5 (mini challenge — LC 1712):** Cho `nums` **không âm**, đếm số cách cắt thành 3 đoạn không rỗng `left, mid, right` sao cho `sum(left) ≤ sum(mid) ≤ sum(right)`. Yêu cầu `O(n log n)`.

In [25]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_cut_positions(s):
    ones = [i for i, c in enumerate(s) if c == "1"]
    k = len(ones) // 3
    cut1, cut2 = [], []
    # TODO: cut1 = các vị trí c thoả ones[k-1] < c <= ones[k]
    # TODO: cut2 = các vị trí c thoả ones[2k-1] < c <= ones[2k]
    return cut1, cut2

In [26]:
# Bài 4 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex4_num_ways_m(s, m):
    # TODO: t % m != 0 -> 0
    # TODO: t == 0 -> C(n-1, m-1) % MOD
    # TODO: nhân các khoảng trống ở ranh giới thứ r = 1..m-1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đếm `t`. (2) Liệt kê `ones`. (3) Nhân `ones[k]-ones[k-1]` với `ones[2k]-ones[2k-1]`.

**Bài 2.** (1) Toàn 0 ⇒ mọi cặp vị trí cắt đều hợp lệ. (2) Có `n-1 = 6` vị trí cắt. (3) `C(6, 2)`.

**Bài 3.** (1) Nhát cắt `c` nghĩa là `s[:c] | s[c:]`. (2) `s[:c]` chứa `k` số `'1'` ⇔ `c` nằm sau `ones[k-1]` và không vượt `ones[k]`. (3) `list(range(ones[k-1] + 1, ones[k] + 1))`.

**Bài 4.** (1) `k = t // m`. (2) Ranh giới thứ `r` nằm giữa `ones[r*k - 1]` và `ones[r*k]`. (3) Nhân dồn và modulo; toàn 0 dùng `math.comb(n-1, m-1)`.

**Bài 5.** (1) Tiền tố `P` không giảm. (2) Với cắt 1 tại `i` (`left = P[i]`), cắt 2 tại `j` cần `P[j] ≥ 2P[i]` và `P[j] ≤ (P[n] + P[i]) / 2`, với `i < j ≤ n-1`. (3) Dùng `bisect_left` / `bisect_right` để đếm số `j`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = Solution().numWays("1100110011")
print("Bài 1:", ans1)
assert ans1 == 3 * 3 == brute_force("1100110011")

# Bài 2
ans2 = Solution().numWays("0000000")
print("Bài 2:", ans2, "= C(6, 2) =", comb(6, 2))
assert ans2 == comb(6, 2) == 15
print("Bài 1, 2: OK")

Bài 1: 9
Bài 2: 15 = C(6, 2) = 15
Bài 1, 2: OK


In [28]:
# Bài 3
def ex3_cut_positions_sol(s):
    ones = [i for i, c in enumerate(s) if c == "1"]
    k = len(ones) // 3
    cut1 = list(range(ones[k - 1] + 1, ones[k] + 1))
    cut2 = list(range(ones[2 * k - 1] + 1, ones[2 * k] + 1))
    return cut1, cut2

def brute_cuts(s):
    n = len(s)
    pairs = [(i, j) for i in range(1, n - 1) for j in range(i + 1, n)
             if s[:i].count("1") == s[i:j].count("1") == s[j:].count("1")]
    return sorted({i for i, _ in pairs}), sorted({j for _, j in pairs})

print("Bài 3:", ex3_cut_positions_sol("100100010100110"))
rng = random.Random(33)
for _ in range(500):
    s = "".join(rng.choice("0001") for _ in range(rng.randint(3, 16)))
    t = s.count("1")
    if t and t % 3 == 0:
        assert ex3_cut_positions_sol(s) == brute_cuts(s), s
print("Bài 3: OK")

Bài 3: ([4, 5, 6, 7], [10, 11, 12])
Bài 3: OK


In [29]:
# Bài 4
def ex4_num_ways_m_sol(s, m):
    ones = [i for i, c in enumerate(s) if c == "1"]
    n, t = len(s), len(ones)
    if t % m:
        return 0
    if t == 0:
        return comb(n - 1, m - 1) % MOD
    k = t // m
    ans = 1
    for r in range(1, m):
        ans = ans * (ones[r * k] - ones[r * k - 1]) % MOD
    return ans

def brute_m(s, m):
    n = len(s)
    def go(start, parts, need):
        if parts == 1:
            return 1 if start < n and s[start:].count("1") == need else 0
        total = 0
        for end in range(start + 1, n):
            if s[start:end].count("1") == need:
                total += go(end, parts - 1, need)
        return total
    t = s.count("1")
    return 0 if t % m else go(0, m, t // m)

rng = random.Random(44)
for _ in range(600):
    s = "".join(rng.choice("001") for _ in range(rng.randint(2, 12)))
    m = rng.randint(2, 4)
    assert ex4_num_ways_m_sol(s, m) == brute_m(s, m) % MOD, (s, m)
assert all(ex4_num_ways_m_sol(s, 3) == Solution().numWays(s) for s, _ in examples + edge)
print("Bài 4: OK, ví dụ:", ex4_num_ways_m_sol("1010101", 4), ex4_num_ways_m_sol("00000", 4))

Bài 4: OK, ví dụ: 8 4


In [30]:
# Bài 5 — LC 1712 (số không âm)
def ex5_ways_to_split(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    ans = 0
    for i in range(1, n - 1):                       # cắt 1 tại i: left = nums[:i]
        lo = max(i + 1, bisect_left(P, 2 * P[i]))
        hi = min(n - 1, bisect_right(P, (P[n] + P[i]) // 2) - 1)
        if hi >= lo:
            ans += hi - lo + 1
    return ans

def ex5_brute(nums):
    n, cnt = len(nums), 0
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            a, b, c = sum(nums[:i]), sum(nums[i:j]), sum(nums[j:])
            if a <= b <= c:
                cnt += 1
    return cnt

print("Bài 5:", ex5_ways_to_split([1, 1, 1]), ex5_ways_to_split([1, 2, 2, 2, 5, 0]), ex5_ways_to_split([3, 2, 1]))
assert ex5_ways_to_split([1, 1, 1]) == 1 and ex5_ways_to_split([1, 2, 2, 2, 5, 0]) == 3 and ex5_ways_to_split([3, 2, 1]) == 0
rng = random.Random(55)
for _ in range(800):
    nums = [rng.randint(0, 5) for _ in range(rng.randint(3, 12))]
    assert ex5_ways_to_split(nums) == ex5_brute(nums), nums
print("Bài 5: OK")

Bài 5: 1 3 0
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Solution:
    def numWays(self, s: str) -> int:
        MOD = 10**9 + 7
        ones = [i for i, c in enumerate(s) if c == "1"]
        t, n = len(ones), len(s)
        if t % 3:
            return 0                                   # không chia đều được
        if t == 0:
            return (n - 1) * (n - 2) // 2 % MOD        # C(n-1, 2)
        k = t // 3
        return (ones[k] - ones[k - 1]) * (ones[2 * k] - ones[2 * k - 1]) % MOD
```

**Template "chia m phần bằng nhau":** kiểm tra chia hết → tiền tố → ranh giới `r` ở chỗ tiền tố = `r·S/m` → (không âm) nhân độ dài đoạn / (có âm) đếm cặp bằng biến `seen`.

**Checklist trước khi submit:**
- [ ] Trả `0` khi `t % 3 != 0`.
- [ ] Nhánh riêng `t == 0`: `C(n-1, 2)` — chia `// 2` **trước** khi `% MOD`.
- [ ] Khoảng trống là `ones[k] - ones[k-1]` (không trừ thêm 1), chỉ số Python bắt đầu từ 0.
- [ ] Ba phần không rỗng: vị trí cắt trong `[1, n-1]`.
- [ ] Lấy `% MOD` ở kết quả cuối.
- [ ] Thử nhanh: `"10101"→4`, `"1001"→0`, `"0000"→3`, `"000"→1`, `"111"→1`.